In [ ]:

!pip install ultralytics -q
!pip install opencv-python matplotlib pandas numpy -q


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import yaml
from pathlib import Path
import torch

print("✓ PyTorch available")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Nếu train trên Kaggle, path sẽ khác
# Giả sử dataset được upload tới Kaggle

KAGGLE_DATASET_PATH = "/kaggle/input/your-dataset-name"  # ⚠️ Thay tên dataset
BASE_DIR = Path(KAGGLE_DATASET_PATH) if Path(KAGGLE_DATASET_PATH).exists() else Path.cwd()

print(f"Base directory: {BASE_DIR}")
print(f"Exists: {BASE_DIR.exists()}")

# Tìm data folder
data_dir = BASE_DIR / "data" if (BASE_DIR / "data").exists() else BASE_DIR
print(f"Data directory: {data_dir}\n")

# Liệt kê folder
for item in data_dir.iterdir():
    print(f"  - {item.name}")

In [ ]:
# Tạo file data.yaml
data_yaml_content = """
path: {path}  # dataset root
train: {train}  # train images (relative to 'path')
val: {val}  # val images
test: {test}  # test images

nc: 1  # number of classes
names: ['Cell']  # class names
"""

# Đường dẫn tương đối
train_path = "images/train"  # Thay đổi theo cấu trúc dataset của bạn
val_path = "images/val"
test_path = "images/test"

# Tạo data.yaml
data_yaml = data_yaml_content.format(
    path=str(data_dir),
    train=train_path,
    val=val_path,
    test=test_path
)

# Lưu file
data_yaml_path = Path("/kaggle/working") / "data.yaml"
with open(data_yaml_path, 'w') as f:
    f.write(data_yaml)

print(f"✓ Created data.yaml")
print(f"\nContent:\n{data_yaml}")

In [ ]:
print("\n" + "="*60)
print("📊 CHECKING DATASET STRUCTURE")
print("="*60)

# Tìm image folders
train_images = list(data_dir.glob(f"{train_path}/*.jpg")) + list(data_dir.glob(f"{train_path}/*.png"))
val_images = list(data_dir.glob(f"{val_path}/*.jpg")) + list(data_dir.glob(f"{val_path}/*.png"))

print(f"\nTrain images: {len(train_images)}")
print(f"Val images: {len(val_images)}")
print(f"Total: {len(train_images) + len(val_images)}")

if len(train_images) > 0:
    print(f"\n✓ Dataset structure looks good!")
else:
    print(f"\n⚠️ No images found! Check data paths.")

In [ ]:
print("\n" + "="*60)
print("🤖 LOADING YOLO MODEL")
print("="*60)

# Load pre-trained model
# Các tùy chọn: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
# n = nano (nhỏ, nhanh), x = xlarge (lớn, chính xác)

model = YOLO('yolov8n.pt')  # Nano version (nhanh, phù hợp cho GPU Kaggle)
print(f"✓ Model loaded: YOLOv8n")
print(f"  Architecture: Nano (lightweight)")
print(f"  Size: ~6.3 MB")

In [ ]:
print("\n" + "="*60)
print("🚀 STARTING YOLO TRAINING")
print("="*60)

# Training parameters
training_params = {
    'data': str(data_yaml_path),     # data.yaml path
    'epochs': 50,                    # số epoch (bạn có thể tăng lên 100)
    'imgsz': 416,                    # kích thước ảnh input
    'batch': 16,                     # batch size (GPU Kaggle có thể chạy 16-32)
    'device': 0,                     # GPU device (0 = GPU đầu tiên)
    'patience': 20,                  # early stopping patience
    'save': True,                    # lưu checkpoint
    'verbose': True,                 # in chi tiết
    'seed': 0,                       # random seed
}

print(f"\nTraining configuration:")
for key, value in training_params.items():
    print(f"  {key}: {value}")

print("\n" + "="*60)
print("Training started... (This may take 10-30 minutes)")
print("="*60 + "\n")

# Train model
results = model.train(**training_params)

print("\n✓ Training completed!")

In [ ]:
print("\n" + "="*60)
print("📊 TRAINING RESULTS")
print("="*60)

# Thông tin về training
runs_dir = Path('runs/detect')
latest_run = sorted(runs_dir.glob('train*'))[-1] if runs_dir.exists() else None

if latest_run:
    print(f"\nLatest training run: {latest_run}")
    print(f"\nGenerated files:")
    for f in latest_run.glob('**/*'):
        if f.is_file():
            print(f"  - {f.relative_to(latest_run)}")
    
    # Metrics file
    metrics_file = latest_run / 'results.csv'
    if metrics_file.exists():
        print(f"\n✓ Metrics saved to: results.csv")

In [ ]:
print("\n" + "="*60)
print("✅ LOADING BEST MODEL")
print("="*60)

# Load best.pt
best_model_path = latest_run / 'weights/best.pt'
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"✓ Best model loaded from: {best_model_path}")
    print(f"  File size: {best_model_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("⚠️ best.pt not found!")
    best_model = model  # Fallback to trained model

In [ ]:
print("\n" + "="*60)
print("🧪 TESTING MODEL ON SAMPLE IMAGES")
print("="*60)

# Predict on validation images
if len(val_images) > 0:
    sample_images = val_images[:3]
    
    for img_path in sample_images:
        results = best_model.predict(str(img_path), conf=0.25, verbose=False)
        
        print(f"\nImage: {img_path.name}")
        print(f"Detections: {len(results[0].boxes)}")
        
        if len(results[0].boxes) > 0:
            for box in results[0].boxes:
                conf = box.conf[0].item()
                print(f"  - Confidence: {conf:.3f}")
else:
    print("No validation images found!")

In [ ]:
import shutil

print("\n" + "="*60)
print("💾 SAVING BEST MODEL")
print("="*60)

# Copy best.pt to /kaggle/working
output_model_path = Path('/kaggle/working') / 'best.pt'
if best_model_path.exists():
    shutil.copy(str(best_model_path), str(output_model_path))
    print(f"\n✓ Model saved to: {output_model_path}")
    print(f"  Size: {output_model_path.stat().st_size / (1024*1024):.2f} MB")
else:
    print("⚠️ Could not find best.pt")